# Analisador de Requisitos de Instauração de PADOs — ANATEL
**Projeto de Mestrado — Framework de Transparência em IA**

Este notebook verifica os **7 REQUISITOS DE CONFORMIDADE da fase de Instauração** em PADOs da ANATEL.
Rule-based Compliance Checking — verificação de conformidade baseada em regras léxicas.

---
### Pré-requisitos
```bash
pip install pymupdf pandas openpyxl
```

In [35]:
# 1.
import re
import fitz  # PyMuPDF
import pandas as pd
from pathlib import Path
from IPython.display import display
import spacy
from sentence_transformers import SentenceTransformer, util
from openpyxl.styles import PatternFill, Font, Alignment

In [ ]:
# 2.
# CONFIGURAÇÃO
PASTA_PADOS = r"diretorio" # Altere aqui o caminho
ARQUIVO_SAIDA = "resultado_requisitos_pados.xlsx"
CHARS_MIN_PAGINA = 100

print(f'Pasta dos PADOs: {Path(PASTA_PADOS).resolve()}')
print(f'Saída: {Path(ARQUIVO_SAIDA).resolve()}')

### Definição dos 7 Requisitos - 
Step A - Definição manual dos Requisitos de Conformidade

Célula 3 - REQUISITOS

Equivale ao Step A do DERECHA: identificação manual dos requisitos normativos (R1-R7) com base legal, padrões léxicos e tipo de documento esperado. No DERECHA isso é feito via análise do GDPR; no nosso caso, via análise do RASA, LGT e Lei 9.784/99.

In [37]:
# 3. REQUISITOS DE CONFORMIDADE (regex)
# Padrões de busca para cada requisito
# Cada item tem: nome, bases_legais, palavras_chave, tipo_documento_esperado
REQUISITOS = {
    "R1_Documento_Motivador": {
        "nome": "Documento Motivador Presente",
        "base_legal": "LGT Art. 173",
        "etapa_esperada": ["Informe", "Relatorio de Fiscalizacao", "Auto de Infracao"],
        "padroes": [
            r"relat[oó]rio\s+de\s+fiscaliza[cç][aã]o",
            r"auto\s+de\s+infra[cç][aã]o",
            r"informe\s+n[oº°]",
            r"solicita[cç][aã]o\s+de\s+fiscaliza[cç][aã]o",
            r"den[uú]ncia",
            r"processo\s+de\s+fiscaliza[cç][aã]o\s+regulat[oó]ria",
        ],
    },
    "R2_Analise_Previa": {
        "nome": "Análise Prévia Realizada",
        "base_legal": "RASA Art. 15",
        "etapa_esperada": ["Informe"],
        "padroes": [
            r"informe\s+n[oº°]\s*\d+",
            r"an[aá]lise\s+(t[eé]cnica|pr[eé]via|dos\s+fatos|da\s+conclus[aã]o)",
            r"ind[íi]cios\s+de\s+(irregularidade|infra[cç][aã]o|descumprimento)",
            r"prop[oõ]e.se\s+a\s+instaurac[aã]o",
            r"descri[cç][aã]o\s+da\s+execu[cç][aã]o",
            r"conclus[aã]o",
        ],
    },
    "R3_Despacho_Instauracao": {
        "nome": "Despacho de Instauração Formal",
        "base_legal": "Lei 9.784/99",
        "etapa_esperada": ["Despacho de Instauracao", "Despacho Ordinatorio"],
        "padroes": [
            r"despacho\s+\S+\s+de\s+instaurac",
            r"instaurar\s+(procedimento|pado|processo)",
            r"resolve.*instaurar",
            r"auto\s+de\s+infra[cç][aã]o\s+n[oº°]\s*[A-Z]{2}\d+",
            r"instaurou.se\s+o\s+pado",
            r"instaurac[aã]o\s+n[oº°]\s*\d+",
        ],
    },
    "R4_Notificacao_Autuado": {
        "nome": "Notificação ao Autuado",
        "base_legal": "CF/88 Art. 5º LV",
        "etapa_esperada": ["Oficio de Notificacao", "Certidao de Intimacao", "Aviso de Recebimento (AR)"],
        "padroes": [
            r"intima.se\s+(essa|a)\s+prestadora",
            r"[oó]ficio\s+n[oº°].*notifica[cç][aã]o",
            r"notifica[cç][aã]o.*instaurac[aã]o",
            r"prazo.*oferecer\s+defesa",
            r"intima.se.*da.*descri[cç][aã]o\s+do\s+fato",
            r"intimac[aã]o.*fiscalizada",
            r"certid[aã]o.*intimac[aã]o",
            r"tipo\s+de\s+intima[cç][aã]o",
            r"instaurac[aã]o\s+de\s+procedimento",
            r"assunto.*intima[cç][aã]o",
            r"intimac[aã]o\s+para\s+oferecimento",
            r"ao\s+representante\s+legal",
            r"aviso\s+de\s+recebimento",
            r"objeto\s+entregue\s+ao\s+destinat[aá]rio",
            r"yo\d{9}br",
        ],
    },
    "R5_Base_Legal": {
        "nome": "Base Legal Citada",
        "base_legal": "Lei 9.784/99 Art. 50",
        "etapa_esperada": ["Despacho de Instauracao", "Oficio de Notificacao"],
        "padroes": [
            r"art\.\s*\d+.*lgt",
            r"resolu[cç][aã]o\s+n[oº°]\s*\d+",
            r"lei\s+n[oº°]\s*\d+",
            r"normas\s+defini[dt]oras\s+da\s+infra[cç][aã]o",
            r"dispositivos?\s*(normativos?|legais?|relacionados?)",
            r"san[cç][oõ]es?\s+aplic[aá]veis?",
        ],
    },
    "R6_Prazo_Defesa": {
        "nome": "Prazo para Defesa Estabelecido",
        "base_legal": "RASA Art. 33",
        "etapa_esperada": ["Oficio de Notificacao"],
        "padroes": [
            r"prazo\s+de\s+1[05]\s+\((?:quinze|dez)\)\s+dias",
            r"15\s+\(quinze\)\s+dias.*defesa",
            r"oferecer\s+defesa.*prazo",
            r"prazo.*apresentar\s+(defesa|alegac[oõ]es)",
        ],
    },
    "R7_Identificacao_Processo": {
        "nome": "Identificação Clara do Processo",
        "base_legal": "Lei 9.784/99",
        "etapa_esperada": ["Qualquer documento"],
        "padroes": [
            r"processo\s+n[oº°]\s*\d{5}\.\d{6}/\d{4}-\d{2}",
            r"cnpj\s+n[oº°]\s*\d{2}\.\d{3}\.\d{3}/\d{4}-\d{2}",
            r"cpf\s+n[oº°]\s*\d{3}\.\d{3}\.\d{3}-\d{2}",
            r"pado\s+n[oº°]\s*\d{5}\.\d{6}/\d{4}-\d{2}",
        ],
    },
}

## Funções de Extração e Análise

Step B - Extração e pré-processamento de texto

Célula 4 - extrair_texto_pdf, detectar_tipo_documento, detectar_tipo_pado

Equivale ao Step B do DERECHA: pipeline NLP de entrada. No DERECHA: tokenização, lematização, POS tagging, dependency parsing. No caso: PyMuPDF para extração, regex para detecção de tipo, normalização de texto.

In [ ]:
# 4. FUNCOES BASE — extração, detecção de tipo, verificação léxica
# Step B do DERECHA: pipeline NLP de entrada
# verificar_requisito_por_etapa e explicar_nao_conforme estao na celula 5
# pois dependem de FRAMES_REQUISITOS e verificar_requisito_semantico

import re
import fitz
import unicodedata
from pathlib import Path

def extrair_texto_pdf(caminho_pdf):
    try:
        doc = fitz.open(str(caminho_pdf))
        texto = ""
        paginas_vazias = 0
        for pagina in doc:
            t = pagina.get_text()
            if len(t.strip()) < CHARS_MIN_PAGINA:
                paginas_vazias += 1
            else:
                texto += t
        doc.close()
        texto = texto.replace('\xa0', ' ')
        texto = re.sub(r' +', ' ', texto)
        return texto, paginas_vazias
    except Exception:
        return "", 0


def detectar_tipo_documento(texto):
    t = texto.lower()
    if re.search(r"despacho\s+ordin[aá]t[oó]rio\s+de\s+instaurac[aã]o", t):
        return "Despacho de Instauração"
    if re.search(r"despacho\s+decis[oó]rio", t):
        return "Despacho Decisório"
    if re.search(r"despacho\s+ordin[aá]t[oó]rio", t):
        return "Despacho Ordinatório"
    if re.search(r"informe\s+n[oº°]", t) and re.search(r"conclus[aã]o|proposta", t):
        return "Informe"
    if re.search(r"relat[oó]rio\s+de\s+fiscaliza[cç][aã]o", t):
        return "Relatório de Fiscalização"
    if re.search(r"auto\s+de\s+infra[cç][aã]o", t) and re.search(r"agente\s+de\s+fiscaliza[cç][aã]o", t):
        return "Auto de Infração"
    if re.search(r"tipo\s+de\s+intima[cç][aã]o", t):
        return "Certidão de Intimação"
    if re.search(r"certid[aã]o\s+de\s+redistribuic[aã]o", t):
        return "Certidão de Redistribuição"
    if re.search(r"certid[aã]o", t):
        return "Certidão"
    if re.search(r"[oó]ficio\s+n[oº°]", t) and re.search(r"intima|notifica|alegac|renuncia|renúncia", t):
        return "Ofício de Notificação"
    if re.search(r"requerimento\s+de\s+informa[cç][oõ]es", t):
        return "Requerimento de Informações"
    if re.search(r"aviso\s+de\s+recebimento", t):
        return "Aviso de Recebimento (AR)"
    if re.search(r"objeto\s+entregue|rastreamento\s+do\s+objeto", t):
        return "Rastreamento Correios"
    if re.search(r"[oó]ficio\s+n[oº°]", t):
        return "Ofício"
    if re.search(r"recurso|peti[cç][aã]o", t):
        return "Petição/Recurso"
    if re.search(r"recibo\s+de\s+peticionamento|usu[aá]rio\s+externo\s+\(signat", t):
        return "Recibo de Peticionamento"
    if re.search(r"termo\s+de\s+(fiscaliza[cç][aã]o|qualifica[cç][aã]o)", t):
        return "Termo de Fiscalização"
    return "Outro"


def detectar_tipo_pado(textos):
    if not textos:
        return "Indeterminado"
    primeiro = textos[0].replace('\xa0', ' ').replace('\n', ' ').lower()
    primeiro = re.sub(r' +', ' ', primeiro)
    if re.search(r"auto\s+de\s+infra", primeiro) and re.search(r"agente\s+de\s+fiscaliza", primeiro):
        return "Tipo B (Auto de Infração em campo)"
    if re.search(r"relatorio\s+de\s+fiscalizacao", primeiro) and not re.search(r"despacho", primeiro):
        return "Tipo B (Auto de Infração em campo)"
    if re.search(r"despacho\s+(ordinat|decis|ordin)", primeiro):
        return "Tipo A (por Informe)"
    if re.search(r"informe\s+n", primeiro):
        return "Tipo A (por Informe)"
    def normalizar(t):
        t = t.replace('\xa0', ' ').replace('\n', ' ')
        t = re.sub(r' +', ' ', t).lower()
        return ''.join(c for c in unicodedata.normalize('NFD', t)
                       if unicodedata.category(c) != 'Mn')
    texto_total = normalizar(" ".join(textos))
    if re.search(r"proposta\s+de\s+instauracao\s+de\s+procedimento", texto_total):
        return "Tipo A (por Informe)"
    if re.search(r"auto\s+de\s+infracao", texto_total) and \
       re.search(r"agente\s+de\s+(telecomunicacoes|fiscalizacao)", texto_total):
        return "Tipo B (Auto de Infração em campo)"
    if len(textos) > 1:
        segundo = textos[1].replace('\xa0', ' ').replace('\n', ' ').lower()
        segundo = re.sub(r' +', ' ', segundo)
        if re.search(r"auto\s+de\s+infra", segundo) and re.search(r"agente\s+de\s+fiscaliza", segundo):
            return "Tipo B (Auto de Infração em campo)"
        if re.search(r"despacho\s+(ordinat|decis|ordin)", segundo):
            return "Tipo A (por Informe)"
    return "Indeterminado"


def verificar_requisito(req_info, texto_total):
    """Verificacao lexica via regex — usada como baseline na celula 9."""
    def remover_acentos(texto):
        return ''.join(
            c for c in unicodedata.normalize('NFD', texto)
            if unicodedata.category(c) != 'Mn'
        )
    t = remover_acentos(texto_total.lower())
    for padrao in req_info["padroes"]:
        padrao_sem_acento = remover_acentos(padrao)
        match = re.search(padrao_sem_acento, t, re.IGNORECASE | re.DOTALL)
        if match:
            inicio = max(0, match.start() - 100)
            fim = min(len(t), match.end() + 100)
            evidencia = " ".join(t[inicio:fim].split())
            return "SIM", evidencia[:250]
    return "NÃO", "—"


def analisar_pado(pasta_pado):
    """Analisa todos os PDFs de uma pasta de PADO."""
    arquivos = sorted(Path(pasta_pado).glob("*.pdf"))
    if not arquivos:
        return None

    documentos = []
    total_paginas_vazias = 0
    for pdf in arquivos:
        texto, pag_vazias = extrair_texto_pdf(pdf)
        tipo = detectar_tipo_documento(texto)
        documentos.append({"arquivo": pdf.name, "tipo": tipo, "texto": texto})
        total_paginas_vazias += pag_vazias

    textos = [d["texto"] for d in documentos]
    texto_meta = " ".join(textos)
    texto_total = " ".join(t.replace('\n', ' ') for t in textos)
    texto_total = re.sub(r' +', ' ', texto_total)

    processo = re.search(r"\d{5}\.\d{6}/\d{4}-\d{2}", texto_meta)
    processo = processo.group() if processo else "N/A"
    autuado = re.search(r"interessado[:\s]+([^\n]{5,60})", texto_meta, re.IGNORECASE)
    autuado = autuado.group(1).strip()[:60] if autuado else "N/A"

    tipo_pado = detectar_tipo_pado(textos)
    tipos_docs = ", ".join(sorted(set(d["tipo"] for d in documentos)))

    resultado = {
        "PADO": Path(pasta_pado).name,
        "Processo": processo,
        "Autuado": autuado,
        "Tipo_PADO": tipo_pado,
        "Qtd_PDFs": len(arquivos),
        "Tipos_Documentos": tipos_docs,
    }

    flag_ocr = "SIM" if total_paginas_vazias > 0 else "NAO"

    # BASELINE LEXICO — regex (Step E versao anterior)
    # Usado como comparacao na celula 9
    # Nao captura variacao semantica — opera sobre forma lexica
    #total_sim = 0
    #for req_id, req_info in REQUISITOS.items():
    #    res, evidencia = verificar_requisito(req_info, texto_total)
    #    resultado[req_id] = res
    #    resultado[f"{req_id}_evidencia"] = evidencia
    #    if res == "SIM":
    #        total_sim += 1

    # SEMANTIC FRAME — sentence-transformers (Step E versao atual)
    # Metodo principal — verifica por etapa processual + matching semantico
    # Captura variacao de linguagem via similaridade de cosseno threshold=0.55
    total_sim = 0
    for req_id, req_info in REQUISITOS.items():
        res, evidencia, etapa_usada = verificar_requisito_por_etapa(
            req_id, req_info, documentos, texto_total
        )
        if res == "NÃO":
            explicacao = explicar_nao_conforme(
                req_id, req_info, documentos, flag_ocr, total_paginas_vazias
            )
        else:
            explicacao = f"Encontrado na etapa: {etapa_usada}"

        resultado[req_id]                  = res
        resultado[f"{req_id}_evidencia"]   = evidencia
        resultado[f"{req_id}_etapa"]       = etapa_usada
        resultado[f"{req_id}_explicacao"]  = explicacao

        if res == "SIM":
            total_sim += 1

    resultado["Conformidade"]      = f"{total_sim}/7"
    resultado["Pct"]               = round(total_sim / 7 * 100)
    resultado["paginas_sem_texto"] = total_paginas_vazias
    resultado["flag_ocr"]          = flag_ocr
    return resultado

print("Funcoes base carregadas — Step B completo")
print("verificar_requisito_por_etapa e explicar_nao_conforme serao carregadas na celula 5")

## Semantics Frames - FRAMES_REQUISITOS 
+ imports spacy/ST
+ embeddings pré-computados
+ função verificar_requisito_semantico
+ função extrair_sentencas

Step C - Definição dos Semantic Frames de referência

Célula 5 - FRAMES_REQUISITOS

Equivale ao Step C do DERECHA: construção manual dos frames semânticos. Cada frame tem verbos-núcleo, objetos-núcleo e sentenças de referência que descrevem o requisito em linguagem natural.

---------------

Step D — Enriquecimento semântico

Célula 5 — bloco de pré-computação de embeddings

Equivale ao Step D do DERECHA: no DERECHA é feito via FrameNet, VerbNet e WordNet. Nocaso é feito via paraphrase-multilingual-MiniLM-L12-v2 - os embeddings das sentenças de referência substituem o enriquecimento lexical do DERECHA.

---------------

Step E — Matching e verificação de conformidade

Célula 5 — verificar_requisito_semantico e célula 4 — verificar_requisito

Equivale ao Step E do DERECHA: comparação entre os frames do documento e os frames dos requisitos. No caso são dois métodos paralelos - regex (léxico) e similaridade de cosseno (semântico).

In [ ]:
# 5. Semantic Frames
nlp = spacy.load('pt_core_news_sm')
modelo_st = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# ─── STEP C: DEFINIÇÃO DOS SEMANTIC FRAMES DOS 7 REQUISITOS ─────────────────
# Cada frame tem:
# - verbos_nucleo: ações que caracterizam o requisito
# - objetos_nucleo: entidades que devem estar presentes
# - descricao: texto semântico para comparação via sentence-transformers
# Atualiza sentenças de referência do R4 e R6 com padrões reais dos PADOs

FRAMES_REQUISITOS = {
    "R1_Documento_Motivador": {
        "nome": "Documento Motivador Presente",
        "base_legal": "LGT Art. 173",
        "verbos_nucleo": ["constatar", "verificar", "identificar", "apurar", "fiscalizar", "denunciar"],
        "objetos_nucleo": ["irregularidade", "indício", "infração", "descumprimento", "irregularidades"],
        "descricao": "fiscalização constatou indícios de irregularidade cometida pela prestadora",
        "sentencas_referencia": [
            "A fiscalização constatou indícios de irregularidade cometida pela prestadora.",
            "O relatório de fiscalização identificou descumprimento de obrigações.",
            "O auto de infração registra a irregularidade verificada em campo.",
            "A denúncia aponta irregularidade no funcionamento da estação.",
        ]
    },
    "R2_Analise_Previa": {
        "nome": "Análise Prévia Realizada",
        "base_legal": "RASA Art. 15",
        "verbos_nucleo": ["propor", "recomendar", "concluir", "analisar", "instaurar"],
        "objetos_nucleo": ["instauração", "procedimento", "pado", "apuração", "proposta"],
        "descricao": "propõe-se a instauração de procedimento para apuração de descumprimento",
        "sentencas_referencia": [
            "Propõe-se a instauração de Procedimento para Apuração de Descumprimento de Obrigações.",
            "Diante do exposto, propõe-se instaurar o PADO em desfavor da prestadora.",
            "A análise conclui pela instauração do procedimento sancionador.",
            "O informe recomenda a abertura de processo administrativo sancionador.",
        ]
    },
    "R3_Despacho_Instauracao": {
        "nome": "Despacho de Instauração Formal",
        "base_legal": "Lei 9.784/99",
        "verbos_nucleo": ["instaurar", "determinar", "resolver", "autuar", "expedir"],
        "objetos_nucleo": ["procedimento", "pado", "processo", "instauração", "apuração"],
        "descricao": "o gerente resolve instaurar o procedimento para apuração de descumprimento de obrigações",
        "sentencas_referencia": [
            "O Gerente resolve instaurar Procedimento para Apuração de Descumprimento de Obrigações.",
            "Determina a instauração de procedimento de apuração em desfavor da prestadora.",
            "O Despacho Ordinatório de Instauração formaliza a abertura do PADO.",
            "Instaura-se o PADO para apuração das irregularidades identificadas.",
        ]
    },
    "R4_Notificacao_Autuado": {
        "nome": "Notificação ao Autuado",
        "base_legal": "CF/88 Art. 5º LV",
        "verbos_nucleo": ["intimar", "notificar", "comunicar", "expedir", "cientificar"],
        "objetos_nucleo": ["intimação", "notificação", "defesa", "prestadora", "autuado"],
        "sentencas_referencia": [
            "Expedir intimação à prestadora para que ofereça defesa no prazo de 15 dias.",
            "Intima-se essa Prestadora da instauração do Pado para apresentar defesa.",
            "Notifica-se o autuado para ciência da instauração do procedimento.",
            "O ofício comunica a prestadora da abertura do processo sancionador.",
            "Nos termos do artigo 82 inciso II do Regimento Interno da Anatel intima-se essa Prestadora.",
            "Ofício de instauração do Procedimento para Apuração de Descumprimento de Obrigações.",
            "A empresa fica intimada da instauração do procedimento administrativo sancionador.",
            "Certidão de cumprimento da intimação eletrônica referente à instauração de procedimento.",
            "Tipo de Intimação Instauração de Procedimento Documento Principal da Intimação Ofício.",
            "Aviso de Recebimento destinatário representante legal prestadora.",
        ]
    },
    "R5_Base_Legal": {
        "nome": "Base Legal Citada",
        "base_legal": "Lei 9.784/99 Art. 50",
        "verbos_nucleo": ["citar", "fundamentar", "dispor", "prever", "estabelecer"],
        "objetos_nucleo": ["artigo", "resolução", "lei", "regulamento", "sanção", "dispositivo"],
        "descricao": "as sanções previstas no artigo do regulamento de aplicação de sanções administrativas",
        "sentencas_referencia": [
            "As sanções previstas no artigo 3º do Regulamento de Aplicação de Sanções Administrativas.",
            "Fundamentado no artigo 173 da Lei Geral de Telecomunicações.",
            "Nos termos do artigo 82 do Regimento Interno da Anatel.",
            "Com base no dispositivo legal do Regulamento Geral de Direitos do Consumidor.",
        ]
    },
    "R6_Prazo_Defesa": {
        "nome": "Prazo para Defesa Estabelecido",
        "base_legal": "RASA Art. 33",
        "verbos_nucleo": ["conceder", "estabelecer", "fixar", "determinar", "oferecer"],
        "objetos_nucleo": ["prazo", "dias", "defesa", "alegações", "manifestação"],
        "sentencas_referencia": [
            "Conceder prazo de 15 quinze dias para apresentação de defesa.",
            "No prazo de 15 dias, ofereça defesa e produza as provas que julgar cabíveis.",
            "Fica estabelecido o prazo de quinze dias para manifestação do autuado.",
            "A prestadora tem 15 dias para apresentar suas alegações e provas.",
            "prazo de 15 quinze dias contados a partir do recebimento para oferecer defesa.",
            "para que no prazo de 15 quinze dias ofereça defesa e produza as provas que julgar cabíveis.",
            "fica essa empresa intimada da abertura do prazo de 10 dez dias para alegações finais.",
            "o interessado tem prazo para apresentar defesa administrativa e produzir provas.",
            "prazo para defesa estabelecido conforme artigo 82 do Regimento Interno da Anatel.",
        ]
    },
    "R7_Identificacao_Processo": {
        "nome": "Identificação Clara do Processo",
        "base_legal": "Lei 9.784/99",
        "verbos_nucleo": ["identificar", "numerar", "registrar", "indicar"],
        "objetos_nucleo": ["processo", "pado", "cnpj", "interessado", "prestadora"],
        "descricao": "processo identificado com número e dados completos do autuado",
        "sentencas_referencia": [
            "Processo nº 53500.102531/2024-47. Interessado: TELEFONICA BRASIL S.A.",
            "O PADO nº 53500.014139/2023-61 é instaurado em desfavor da Telefônica Brasil S.A., CNPJ nº 02.558.157/0001-62.",
            "Identificado o processo com número SEI e dados completos da autuada.",
        ]
    },
}

# ── STEP D: Pré-computa embeddings das sentenças de referência ───────────────
print("Pré-computando embeddings...")
for req_id, frame in FRAMES_REQUISITOS.items():
    frame["embeddings_referencia"] = modelo_st.encode(
        frame["sentencas_referencia"],
        convert_to_tensor=True
    )
print("Embeddings computados")

# ── STEP E: Funções de matching semântico ────────────────────────────────────
def extrair_sentencas(texto, max_sentencas=50):
    texto = texto.replace('\xa0', ' ').replace('\n', ' ')
    texto = re.sub(r' +', ' ', texto)
    sentencas = re.split(r'(?<=[.!?])\s+', texto)
    sentencas = [s.strip() for s in sentencas if 15 < len(s.strip()) < 500]
    return sentencas[:max_sentencas]


def verificar_requisito_semantico(req_id, frame, texto_total, threshold=0.55):
    sentencas = extrair_sentencas(texto_total)
    if not sentencas:
        return "NÃO", 0.0, "—"
    embeddings_doc = modelo_st.encode(sentencas, convert_to_tensor=True)
    scores = util.cos_sim(frame["embeddings_referencia"], embeddings_doc)
    score_max = float(scores.max())
    idx_doc = int(scores.argmax() % len(sentencas))
    sentenca_match = sentencas[idx_doc] if idx_doc < len(sentencas) else "—"
    resultado = "SIM" if score_max >= threshold else "NÃO"
    return resultado, round(score_max, 3), sentenca_match[:200]


print("Semantic Frames e matching semântico prontos")
print(f"7 frames definidos | threshold={0.55} | modelo=multilingual-MiniLM-L12-v2")

In [ ]:
# ── STEP E2: Verificação por etapa e explicação granular ─────────────────────
# Entregas 1 e 2 do orientador
# Dependem de FRAMES_REQUISITOS e verificar_requisito_semantico
# Por isso ficam aqui na célula 5 e não na célula 4

def explicar_nao_conforme(req_id, req_info, documentos, flag_ocr, paginas_vazias):
    """
    Entrega 2: explica por que um requisito retornou NAO.
    Identifica a causa raiz entre tres possibilidades.
    """
    etapas_esperadas = set(req_info.get("etapa_esperada", []))
    etapas_presentes = set(d["tipo"] for d in documentos)

    # Causa 1: PDF escaneado
    if flag_ocr == "SIM" and paginas_vazias > 0:
        return f"PDF com {paginas_vazias} paginas sem texto extraivel — possivel documento escaneado"

    # Causa 2: etapa processual ausente
    if "Qualquer documento" not in etapas_esperadas:
        faltando = etapas_esperadas - etapas_presentes
        if faltando:
            return f"Etapa processual ausente na pasta: {', '.join(sorted(faltando))}"

    # Causa 3: etapa presente mas requisito nao encontrado pelo SF
    return "Documento da etapa esperada presente mas requisito nao localizado pelo Semantic Frame"


def verificar_requisito_por_etapa(req_id, req_info, documentos, texto_total):
    """
    Entrega 1: verifica o requisito no documento da etapa correta.
    Metodo principal: Semantic Frame via sentence-transformers.
    Retorna (resultado, sentenca_match, etapa_onde_encontrou)
    """
    etapas_esperadas = set(req_info.get("etapa_esperada", []))
    frame = FRAMES_REQUISITOS[req_id]

    # R7 e qualquer requisito sem etapa definida: SF no texto completo
    if "Qualquer documento" in etapas_esperadas or not etapas_esperadas:
        res, score, sentenca = verificar_requisito_semantico(req_id, frame, texto_total)
        return res, sentenca, f"texto completo SF (score={score})"

    # Filtra documentos da etapa esperada
    docs_etapa = [d for d in documentos if d["tipo"] in etapas_esperadas]

    if not docs_etapa:
        # Fallback: SF no texto completo se etapa nao encontrada na pasta
        res, score, sentenca = verificar_requisito_semantico(req_id, frame, texto_total)
        return res, sentenca, f"fallback SF (score={score})"

    # Verifica nos documentos da etapa correta via SF
    texto_etapa = " ".join(d["texto"] for d in docs_etapa)
    res, score, sentenca = verificar_requisito_semantico(req_id, frame, texto_etapa)
    return res, sentenca, f"{docs_etapa[0]['tipo']} (SF, score={score})"


print("verificar_requisito_por_etapa e explicar_nao_conforme prontos")
print("Pipeline completo: Step B (celula 4) + Step C/D/E (celula 5)")

## Análise em Todos os PADOs

In [ ]:
# 6. Rodar análise (gera resultados)
pasta_raiz = Path(PASTA_PADOS)
subpastas = sorted([p for p in pasta_raiz.iterdir() if p.is_dir()])

print(f'{len(subpastas)} pastas de PADO encontradas:\n')
for p in subpastas:
    pdfs = list(p.glob('*.pdf'))
    print(f'  {p.name} ({len(pdfs)} PDFs)')

print('\nProcessando...')
resultados = []
for pasta in subpastas:
    r = analisar_pado(pasta)
    if r:
        resultados.append(r)
        sim = sum(1 for k in REQUISITOS if r[k] == 'SIM')
        print(f'{r["PADO"][:45]} → {r["Conformidade"]} | {r["Tipo_PADO"]}')

print(f'\n{len(resultados)} PADOs analisados.')

In [ ]:
# 7. Célula para corrigir manualmente resultados que o script errou
# Exemplo: o R3 do PADO Quijingue é PARCIAL, não NÃO

# Sintaxe: correcoes["nome_da_pasta"]["R3_Despacho_Instauracao"] = "PARCIAL"
correcoes = {
    "53524.000191_2022-81": {
        "Autuado": "— PDF escaneado, OCR ilegível —",
        "R1_Documento_Motivador": "SIM",   # Auto de Infração MG20220120l013 confirmado visualmente
        "obs": "PDFs escaneados — verificação manual necessária"
    },
    "53524.000482_2022-79": {
        "Autuado": "Associação Cultural de Dom Cavati - Rádio Cidade FM",
    },
}

for r in resultados:
    if r['PADO'] in correcoes:
        for campo, valor in correcoes[r['PADO']].items():
            r[campo] = valor
            print(f"{r['PADO']} → {campo} = {valor}")

print('Correções aplicadas (rode Célula de Matriz de Conformidade novamente para ver o resultado atualizado)')

## Visualizar Matriz de Conformidade

In [ ]:
# 8.
import pandas as pd

# Colunas da matriz resumida
cols_resumo = ["PADO", "Processo", "Autuado", "Tipo_PADO", "Conformidade", "Pct"] + list(REQUISITOS.keys())
df = pd.DataFrame(resultados)[cols_resumo]

# Renomeia colunas de requisito para nome curto
nomes_curtos = {k: f"R{i+1}" for i, k in enumerate(REQUISITOS.keys())}
df_display = df.rename(columns=nomes_curtos)

# Estiliza SIM/NÃO/PARCIAL com cores
def colorir(val):
    if val == 'SIM':
        return 'background-color: #C6EFCE; color: #276221; font-weight: bold'
    elif val == 'NÃO':
        return 'background-color: #FFC7CE; color: #9C0006; font-weight: bold'
    elif val == 'PARCIAL':
        return 'background-color: #FFEB9C; color: #9C6500; font-weight: bold'
    return ''

cols_req = [f"R{i+1}" for i in range(7)]
styled = df_display.style.applymap(colorir, subset=cols_req)

print('Legenda: R1=Documento Motivador | R2=Análise Prévia | R3=Despacho Instauração')
print('         R4=Notificação Autuado | R5=Base Legal | R6=Prazo Defesa | R7=Identificação')
print()
display(styled)

identificado doi pados contendo imagens, o que causou um problema na leitura do campo Autuado
PADO 12 (53524.000191_2022-81): O PDF é escaneado com OCR ruim — o texto saiu todo corrompido com caracteres estranhos (¬z..~.z.z..›«.). O script não consegue extrair texto limpo.
PADO 13 (53524.000482_2022-79): O autuado está em um Recibo de Peticionamento (doc_8039909.pdf) como "Associação Cultural de Dom Cavati - Rádio Cidade FM", não no campo "Interessado:" padrão.

inclui abaixo um condigo para pegar essas informações que vou incluir no codigo um tratamento para esses casos.

Isso levanta também um ponto metodológico importante para a dissertação: PDFs escaneados são um caso limite do framework — o pipeline NLP não consegue processar imagem, só texto. Precisar mencionar isso como uma limitação e possivelmente incluir uma etapa de OCR (como o pytesseract) para tratar esses casos.

## Comparação Regex vs Semântico

In [ ]:
# 9.
print("Comparando REGEX vs SEMÂNTICO nos 104 PADOs\n")
print(f"{'PADO':<25} {'Requisito':<30} {'REGEX':<8} {'SEMÂNTICO':<12} {'Score':<7} {'Status'}")
print("─" * 95)

divergencias = []
total_verificacoes = 0
total_concordancias = 0

for r in resultados:
    pasta = Path(PASTA_PADOS) / r['PADO']
    texto = " ".join(extrair_texto_pdf(pdf)[0] for pdf in sorted(pasta.glob("*.pdf")))

    for req_id, frame in FRAMES_REQUISITOS.items():
        res_regex = r[req_id]
        res_sem, score, sentenca = verificar_requisito_semantico(req_id, frame, texto)

        total_verificacoes += 1
        concordou = res_regex == res_sem
        if concordou:
            total_concordancias += 1
        else:
            divergencias.append({
                "PADO": r['PADO'], "Requisito": req_id,
                "Regex": res_regex, "Semantico": res_sem,
                "Score": score, "Sentenca": sentenca
            })
            status = "DIVERGE"
            print(f"{r['PADO'][:24]:<25} {req_id[:29]:<30} {res_regex:<8} {res_sem:<12} {score:.3f}  {status}")

print(f"\n{'─'*95}")
print(f"Total de verificações: {total_verificacoes}")
print(f"Concordâncias:         {total_concordancias}/{total_verificacoes} ({total_concordancias/total_verificacoes*100:.1f}%)")
print(f"Divergências:          {len(divergencias)}/{total_verificacoes} ({len(divergencias)/total_verificacoes*100:.1f}%)")

In [ ]:
# 10.
cols_resumo = ["PADO", "Processo", "Autuado", "Tipo_PADO", "Conformidade", "Pct", "paginas_sem_texto", "flag_ocr"] + list(REQUISITOS.keys())
df_resumo = pd.DataFrame(resultados)[cols_resumo]
df_detalhe = pd.DataFrame(resultados)

with pd.ExcelWriter(ARQUIVO_SAIDA, engine='openpyxl') as writer:
    df_resumo.to_excel(writer, sheet_name='Matriz de Conformidade', index=False)
    df_detalhe.to_excel(writer, sheet_name='Detalhes e Evidências', index=False)

    ws = writer.sheets['Matriz de Conformidade']
    azul    = PatternFill(start_color='2E5090', end_color='2E5090', fill_type='solid')
    verde   = PatternFill(start_color='C6EFCE', end_color='C6EFCE', fill_type='solid')
    vermelho = PatternFill(start_color='FFC7CE', end_color='FFC7CE', fill_type='solid')
    amarelo = PatternFill(start_color='FFEB9C', end_color='FFEB9C', fill_type='solid')

    for cell in ws[1]:
        cell.fill = azul
        cell.font = Font(color='FFFFFF', bold=True)
        cell.alignment = Alignment(horizontal='center', wrap_text=True)

    for row in ws.iter_rows(min_row=2):
        for cell in row:
            if cell.value == 'SIM':
                cell.fill = verde
                cell.alignment = Alignment(horizontal='center')
            elif cell.value == 'NÃO':
                cell.fill = vermelho
                cell.alignment = Alignment(horizontal='center')
            elif cell.value == 'PARCIAL':
                cell.fill = amarelo
                cell.alignment = Alignment(horizontal='center')

    for col in ws.columns:
        ws.column_dimensions[col[0].column_letter].width = min(
            max(len(str(c.value or '')) for c in col) + 4, 35
        )

print(f' Planilha salva em: {Path(ARQUIVO_SAIDA).resolve()}')

In [ ]:
# 11.
print('=' * 55)
print('ESTATÍSTICAS GERAIS')
print('=' * 55)
print(f'Total de PADOs analisados: {len(resultados)}')
print(f'Tipo A (por Informe):      {sum(1 for r in resultados if "Tipo A" in r["Tipo_PADO"])}')
print(f'Tipo B (Auto de Infração): {sum(1 for r in resultados if "Tipo B" in r["Tipo_PADO"])}')
print(f'Legado:                    {sum(1 for r in resultados if "Legado" in r["Tipo_PADO"])}')
print(f'Indeterminado:             {sum(1 for r in resultados if "Indeterminado" in r["Tipo_PADO"])}')
print()
print('Frequência por Requisito:')
print('-' * 55)
for i, (req_id, req_info) in enumerate(REQUISITOS.items(), 1):
    sim = sum(1 for r in resultados if r[req_id] == 'SIM')
    nao = sum(1 for r in resultados if r[req_id] == 'NÃO')
    total = len(resultados)
    pct = sim / total * 100
    barra = '█' * int(pct / 5) + '░' * (20 - int(pct / 5))
    print(f'R{i} {req_info["nome"][:28]:28s} {barra} {pct:5.1f}% ({sim}/{total})')